# Biomarker Analysis Pipeline (v3)

1:1 line-matched ICI vs never-ICI cohorts with two propensity score models.

### Matching
- **1:1** — one control per ICI case, matched on (cancer_type, line_category)

### Propensity score models
- **embeddings_only** — elastic net CV LR on text embeddings
- **all_covariates** — elastic net CV LR on embeddings + demographics + cancer type + panel version + line

### Analysis tracks
- **Track 1** — ICI-only, prognostic: `S(t) ~ base_vars + line_dummies + marker`
  - **unweighted** + **ATE** (1/ps generalizability weights)
- **Track 2** — Full cohort, predictive interaction: `S(t) ~ base_vars + line_dummies + marker + ICI + marker×ICI`
  - **noIPTW** + **ATE** weights

### Robustness filtering
- **Track 1**: ≥3 of 4 schemes (2 weights × 2 PS models) with consistent HR direction
- **Track 2**: ≥3 of 4 schemes (2 weights × 2 PS models) with consistent classifier

### Stages
1. **Data regeneration** — `generate_all_non_text_covariates.py`
2. **Line-matched cohorts** — `build_line_matched_cohort.py`
3. **Propensity scores** — `ICI_LRs.py` (elastic net CV)
4. **IPTW datasets** — `generate_IPTW_df.py --ps_model {embeddings_only,all_covariates}`
5. **Cox models** — `run_IPTW_analysis.py --ps_model {embeddings_only,all_covariates}`
6. **Compile results** — cross-scheme robustness filtering across Tracks 1 and 2

In [ ]:
import subprocess
import sys
import os

SCRIPT_DIR = os.path.dirname(os.path.abspath('__file__'))
PS_MODELS = ['embeddings_only', 'all_covariates']

def run_and_stream(label, cmd):
    """Run a command and stream its output inline."""
    print(f"\n--- {label} ---")
    result = subprocess.run(cmd, cwd=SCRIPT_DIR,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            universal_newlines=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"FAILED (exit code {result.returncode})")
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"{label} failed")
    print(f"Done: {label}")

## Stage 1: Data Regeneration (SV/Fusion fix)

Re-run `generate_all_non_text_covariates.py` to ensure `complete_somatic_data_df.csv` includes correct SV/Fusion columns.

In [ ]:
run_and_stream('Data regeneration',
               [sys.executable, '../data_preprocessing/generate_all_non_text_covariates.py'])

## Stage 2: Line-Matched Cohort Construction

Build 1:1 and 1:k matched cohorts on (cancer_type, line_category).

In [ ]:
run_and_stream('Line-matched cohorts',
               [sys.executable, 'build_line_matched_cohort.py'])

## Stage 3: Propensity Score Generation

Train embeddings-only and all-covariates elastic net CV logistic regression propensity models.

In [ ]:
run_and_stream('Propensity scores',
               [sys.executable, 'ICI_LRs.py'])

## Stage 4: IPTW Dataset Generation

Build IPTW datasets for each {matching, ps_model} combination.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'IPTW dataset ({ps_model})',
                   [sys.executable, 'generate_IPTW_df.py', '--ps_model', ps_model])

## Stage 5: Cox Model Analysis

Run Track 1 (ICI-only, ATE generalizability-weighted + unweighted) and Track 2 (full-cohort interaction, ATE + unweighted) for each PS model.

In [ ]:
for ps_model in PS_MODELS:
    run_and_stream(f'Cox models ({ps_model})',
                   [sys.executable, 'run_IPTW_analysis.py', '--ps_model', ps_model])

## Stage 6: Compile Results

Aggregate significant hits across all specifications and apply cross-scheme robustness filtering.

In [ ]:
import re
import numpy as np
import pandas as pd

OUTPUT_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/biomarker_analysis/'
COMPILED_PATH = os.path.join(OUTPUT_PATH, 'compiled_results/')
os.makedirs(COMPILED_PATH, exist_ok=True)

MATCHINGS = ['1to1']
MIN_SCHEMES_TRACK1 = 3  # out of 4 (2 weights x 2 PS models = 75%)
MIN_SCHEMES_TRACK2 = 3  # out of 4 (2 weights x 2 PS models = 75%)

# Track 1: ICI-only (ATE generalizability-weighted + unweighted)
TRACK1_WEIGHTS = ['unweighted', 'ATE']
# Track 2: full-cohort interaction (ATE + unweighted)
TRACK2_WEIGHTS = ['ATE', 'noIPTW']

# Discover cancer types from result filenames
cancer_types = set()
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for fname in os.listdir(run_path):
            m = re.match(r'(.+)_track[12]_', fname)
            if m:
                cancer_types.add(m.group(1))
cancer_types = sorted(cancer_types)
print(f"Discovered cancer types: {cancer_types}")

# ================================================
# 1. Compile all significant hits
# ================================================

def _collect_track1_hits(weight_list):
    """Collect significant Track 1 hits."""
    rows = []
    for matching in MATCHINGS:
        for ps_model in PS_MODELS:
            spec = f'{matching}_{ps_model}'
            run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
            for ct in cancer_types:
                for weight_type in weight_list:
                    fname = os.path.join(run_path, f'{ct}_track1_{weight_type}_ICI_only.csv')
                    if not os.path.exists(fname):
                        continue
                    df = pd.read_csv(fname)
                    if 'significant_marker' in df.columns:
                        hits = df.loc[df['significant_marker']].copy()
                        hits['matching'] = matching
                        hits['ps_model'] = ps_model
                        hits['weight_type'] = weight_type
                        hits['cancer_type'] = ct
                        rows.append(hits)
    compiled = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return compiled

t1_compiled = _collect_track1_hits(TRACK1_WEIGHTS)
t1_compiled.to_csv(os.path.join(COMPILED_PATH, 'track1_all_significant_hits.csv'), index=False)
print(f"Track 1 (ICI-only): {len(t1_compiled)} significant hits across all specs")

# --- Track 2: compile interaction hits ---
track2_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        for ct in cancer_types:
            for weight_type in TRACK2_WEIGHTS:
                fname = os.path.join(run_path, f'{ct}_track2_{weight_type}_interaction.csv')
                if not os.path.exists(fname):
                    continue
                df = pd.read_csv(fname)
                if 'significant_predictive' in df.columns:
                    hits = df.loc[df['significant_predictive']].copy()
                    hits['matching'] = matching
                    hits['ps_model'] = ps_model
                    hits['weight_type'] = weight_type
                    hits['cancer_type'] = ct
                    track2_rows.append(hits)

t2_compiled = pd.concat(track2_rows, ignore_index=True) if track2_rows else pd.DataFrame()
t2_compiled.to_csv(os.path.join(COMPILED_PATH, 'track2_all_significant_hits.csv'), index=False)
print(f"Track 2 (interaction): {len(t2_compiled)} significant hits across all specs")

# ================================================
# 2. Cross-scheme robustness filtering
# ================================================

def cross_scheme_filter_track1(df, min_schemes=2):
    """Keep markers significant in >= min_schemes with consistent HR direction."""
    if df.empty:
        return df
    df = df.copy()
    df['scheme'] = df['matching'] + '|' + df['ps_model'] + '|' + df['weight_type']
    grouped = df.groupby(['marker', 'cancer_type'])

    robust = []
    for (marker, ct), grp in grouped:
        n_schemes = grp['scheme'].nunique()
        if n_schemes < min_schemes:
            continue
        all_risk = (grp['HR_marker'] > 1).all()
        all_prot = (grp['HR_marker'] < 1).all()
        if not (all_risk or all_prot):
            continue
        if 'extreme_hr_flag' in grp.columns and grp['extreme_hr_flag'].any():
            continue
        robust.append({
            'marker': marker,
            'cancer_type': ct,
            'n_schemes': n_schemes,
            'direction': 'risk' if all_risk else 'protective',
            'HR_median': grp['HR_marker'].median(),
            'HR_min': grp['HR_marker'].min(),
            'HR_max': grp['HR_marker'].max(),
            'FDR_min': grp['FDR_marker'].min(),
            'FDR_max': grp['FDR_marker'].max(),
            'mutation_type': grp['mutation_type'].iloc[0],
        })
    return pd.DataFrame(robust)


def cross_scheme_filter_track2(df, min_schemes=2):
    """Keep markers significant in >= min_schemes with consistent classifier."""
    if df.empty:
        return df
    df = df.copy()
    df['scheme'] = df['matching'] + '|' + df['ps_model'] + '|' + df['weight_type']
    grouped = df.groupby(['marker', 'cancer_type'])

    robust = []
    for (marker, ct), grp in grouped:
        n_schemes = grp['scheme'].nunique()
        if n_schemes < min_schemes:
            continue
        classifiers = grp['classifier'].unique()
        if len(classifiers) > 1:
            continue
        if 'extreme_hr_flag' in grp.columns and grp['extreme_hr_flag'].any():
            continue
        has_inf = (~np.isfinite(grp['HR_markerxICI'])).any()
        if has_inf:
            continue

        sig_ici_count = (grp.get('significant_in_ICI', pd.Series(dtype=bool)) == True).sum()

        row = {
            'marker': marker,
            'cancer_type': ct,
            'n_schemes': n_schemes,
            'classifier': classifiers[0],
            'FDR_min': grp['FDR_markerxICI'].min(),
            'FDR_max': grp['FDR_markerxICI'].max(),
            'HR_ICI_median': grp['HR_marker_ICI'].median(),
            'HR_nonICI_median': grp['HR_marker_nonICI'].median(),
            'sig_in_ICI_count': sig_ici_count,
            'mutation_type': grp['mutation_type'].iloc[0],
        }

        for ec in ['n_ICI_pos', 'events_ICI_pos', 'n_nonICI_pos', 'events_nonICI_pos']:
            if ec in grp.columns:
                row[ec + '_median'] = grp[ec].median()

        robust.append(row)
    return pd.DataFrame(robust)


t1_robust = cross_scheme_filter_track1(t1_compiled, min_schemes=MIN_SCHEMES_TRACK1)
t2_robust = cross_scheme_filter_track2(t2_compiled, min_schemes=MIN_SCHEMES_TRACK2)

print(f"\nTrack 1 cross-scheme robust (>={MIN_SCHEMES_TRACK1} of 4 schemes): {len(t1_robust)}")
print(f"Track 2 cross-scheme robust (>={MIN_SCHEMES_TRACK2} of 4 schemes): {len(t2_robust)}")

t1_robust.to_csv(os.path.join(COMPILED_PATH, 'track1_cross_scheme_robust.csv'), index=False)
t2_robust.to_csv(os.path.join(COMPILED_PATH, 'track2_cross_scheme_robust.csv'), index=False)

# ================================================
# 3. Diagnostics summary
# ================================================
diag_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for ct in cancer_types:
            diag_path = os.path.join(run_path, f'{ct}_diagnostics/')
            ess_file = os.path.join(diag_path, 'effective_sample_sizes.csv')
            if os.path.isfile(ess_file):
                ess = pd.read_csv(ess_file)
                ess['matching'] = matching
                ess['ps_model'] = ps_model
                diag_rows.append(ess)

if diag_rows:
    diag_df = pd.concat(diag_rows, ignore_index=True)
    diag_df.to_csv(os.path.join(COMPILED_PATH, 'scheme_diagnostics_summary.csv'), index=False)
    print(f"\nDiagnostics summary saved ({len(diag_df)} rows)")

# ================================================
# 4. Patient counts per track
# ================================================
count_rows = []
for matching in MATCHINGS:
    for ps_model in PS_MODELS:
        spec = f'{matching}_{ps_model}'
        run_path = os.path.join(OUTPUT_PATH, f'IPTW_runs_{spec}/')
        if not os.path.isdir(run_path):
            continue
        for ct in cancer_types:
            ess_file = os.path.join(run_path, f'{ct}_diagnostics/',
                                    'effective_sample_sizes.csv')
            if not os.path.isfile(ess_file):
                continue
            ess = pd.read_csv(ess_file)
            row = ess.iloc[0] if len(ess) else None
            if row is None:
                continue

            n_ici = int(row.get('N_treated', 0))
            n_ctrl = int(row.get('N_control', 0))
            ev_ici = int(row.get('events_treated', 0))
            ev_ctrl = int(row.get('events_control', 0))

            t1_markers = len(t1_compiled[
                (t1_compiled['cancer_type'] == ct) &
                (t1_compiled['matching'] == matching) &
                (t1_compiled['ps_model'] == ps_model)
            ]) if not t1_compiled.empty else 0

            t2_markers = len(t2_compiled[
                (t2_compiled['cancer_type'] == ct) &
                (t2_compiled['matching'] == matching) &
                (t2_compiled['ps_model'] == ps_model)
            ]) if not t2_compiled.empty else 0

            count_rows.append({
                'cancer_type': ct,
                'matching': matching,
                'ps_model': ps_model,
                'n_ICI': n_ici,
                'n_nonICI': n_ctrl,
                'n_total': n_ici + n_ctrl,
                'events_ICI': ev_ici,
                'events_nonICI': ev_ctrl,
                'event_rate_ICI': ev_ici / max(n_ici, 1),
                'event_rate_nonICI': ev_ctrl / max(n_ctrl, 1),
                'track1_sig_hits': t1_markers,
                'track2_sig_hits': t2_markers,
            })

if count_rows:
    counts_df = pd.DataFrame(count_rows)
    counts_df.to_csv(os.path.join(COMPILED_PATH, 'patient_counts_by_track.csv'), index=False)
    print(f"\nPatient counts saved ({len(counts_df)} rows)")

print(f"\nAll outputs saved to {COMPILED_PATH}")